In [0]:
from pyspark.sql.functions import *

recommendations = spark.table(
    "agentdb.intelligence.recommendation_registry"
)

inventory_risk = spark.table(
    "agentdb.intelligence.inventory_risk"
)

supplier_risk = spark.table(
    "agentdb.intelligence.supplier_risk"
)

In [0]:
outcome_base = (

    recommendations.alias("r")

    .join(
        inventory_risk.alias("i"),
        [
            "product_key",
            "store_key"
        ],
        "left"
    )

    .join(
        supplier_risk.alias("s"),
        "supplier_key",
        "left"
    )
)

In [0]:
outcomes = (

    outcome_base

    .withColumn(

        "stockout_prevented",

        when(

            (
                col("recommendation_type") == "REORDER"
            )

            &

            (
                col("projected_days_to_stockout") < 30
            ),

            True

        ).when(

            (
                col("recommendation_type") == "EXPEDITE_PO"
            )

            &

            (
                col("projected_days_to_stockout") < 14
            ),

            True

        ).otherwise(False)
    )

    .withColumn(

        "inventory_reduced",

        when(
            col("recommendation_type") == "REORDER",
            True
        ).otherwise(False)
    )

    .withColumn(

        "supplier_issue_mitigated",

        when(

            (
                col("recommendation_type") == "SUPPLIER_ALERT"
            )

            &

            (
                col("supplier_risk_score") > 0.60
            ),

            True

        ).otherwise(False)
    )
)

In [0]:
outcomes = (

    outcomes

    .withColumn(

        "outcome_status",

        when(
            col("stockout_prevented") == True,
            "SUCCESS"
        )

        .when(
            col("supplier_issue_mitigated") == True,
            "SUCCESS"
        )

        .otherwise(
            "NO_IMPACT"
        )
    )
)

In [0]:
outcomes = (

    outcomes

    .withColumn(

        "outcome_reason",

        when(
            col("stockout_prevented") == True,
            "Recommendation likely prevents projected stockout"
        )

        .when(
            col("supplier_issue_mitigated") == True,
            "Recommendation highlights high-risk supplier"
        )

        .otherwise(
            "No measurable impact detected"
        )
    )
)

In [0]:
recommendation_outcomes = (

    outcomes

    .select(

        "recommendation_id",

        "recommendation_type",

        "outcome_status",

        "outcome_reason",

        "stockout_prevented",

        "inventory_reduced",

        "supplier_issue_mitigated"

    )

    .withColumn(
        "evaluated_at",
        current_timestamp()
    )
)

In [0]:
(
    recommendation_outcomes
    .select(
        "recommendation_id",
        "outcome_status",
        "outcome_reason",
        "stockout_prevented",
        "inventory_reduced",
        "evaluated_at"
    )

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "agentdb.intelligence.recommendation_outcome"
    )
)

In [0]:
evaluation = (

    recommendation_outcomes

    .agg(

        count("*")
        .alias(
            "total_recommendations"
        ),

        sum(
            when(
                col("recommendation_type") == "REORDER",
                1
            ).otherwise(0)
        ).alias(
            "reorder_recommendations"
        ),

        sum(
            when(
                col("recommendation_type") == "EXPEDITE_PO",
                1
            ).otherwise(0)
        ).alias(
            "expedite_recommendations"
        ),

        sum(
            when(
                col("recommendation_type") == "SUPPLIER_ALERT",
                1
            ).otherwise(0)
        ).alias(
            "supplier_alerts"
        ),

        sum(
            when(
                col("stockout_prevented"),
                1
            ).otherwise(0)
        ).alias(
            "stockouts_prevented"
        ),

        sum(
            when(
                col("outcome_status") == "NO_IMPACT",
                1
            ).otherwise(0)
        ).alias(
            "false_positive_recommendations"
        )
    )

    .withColumn(
        "evaluation_date",
        current_date()
    )
)

In [0]:
evaluation = (

    evaluation

    .withColumn(

        "precision",

        when(
            col("total_recommendations") > 0,

            col("stockouts_prevented")
            /
            col("total_recommendations")
        )
    )

    .withColumn(

        "recall",

        when(
            col("stockouts_prevented") > 0,

            col("stockouts_prevented")
            /
            col("stockouts_prevented")
        ).otherwise(0.0)
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )
)

In [0]:
(
    evaluation

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "agentdb.evaluation.recommendation_metrics"
    )
)

In [0]:
display(
    recommendation_outcomes
    .groupBy("outcome_status")
    .count()
)

In [0]:
display(
    spark.table(
        "agentdb.evaluation.recommendation_metrics"
    )
)